# 04 — 2013 local-staging I/O benchmark

This notebook benchmarks a bounded, complete annual data
path before a training framework or permanent shard format
is selected.

It copies the 2013 archive from Drive to Colab-local
storage, safely extracts its NetCDF members, constructs all
canonical 2013 DBZ/VEL frame tensors, verifies every label,
records throughput, and deletes all temporary artifacts.

No tensors or derived training data are written to Drive.


In [1]:
from google.colab import drive

drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from pathlib import Path

PACKAGE_VERSION = "0.1.6"
BACKUP_ROOT = Path(
    "/content/drive/MyDrive/TorNet_Backup"
)
PACKAGE_PATH = (
    BACKUP_ROOT
    / "packages"
    / (
        "tornet_detection-"
        f"{PACKAGE_VERSION}-py3-none-any.whl"
    )
)
MANIFESTS_ROOT = BACKUP_ROOT / "manifests"
DRIVE_ARCHIVE_PATH = (
    BACKUP_ROOT / "tornet_2013.tar.gz"
)

LOCAL_ARCHIVE_PATH = Path(
    "/content/tornet_2013.tar.gz"
)
EXTRACTION_ROOT = Path(
    "/content/tornet_2013_extracted"
)

required_paths = [
    PACKAGE_PATH,
    MANIFESTS_ROOT,
    DRIVE_ARCHIVE_PATH,
]

missing = [
    str(path)
    for path in required_paths
    if not path.exists()
]

if missing:
    raise FileNotFoundError(
        "Required Drive paths are missing: "
        + ", ".join(missing)
    )

print("package:", PACKAGE_PATH)
print("archive:", DRIVE_ARCHIVE_PATH)
print("manifests:", MANIFESTS_ROOT)


package: /content/drive/MyDrive/TorNet_Backup/packages/tornet_detection-0.1.6-py3-none-any.whl
archive: /content/drive/MyDrive/TorNet_Backup/tornet_2013.tar.gz
manifests: /content/drive/MyDrive/TorNet_Backup/manifests


In [3]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "netCDF4>=1.7",
    ],
    check=True,
)

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-deps",
        "--force-reinstall",
        str(PACKAGE_PATH),
    ],
    check=True,
)


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '--no-deps', '--force-reinstall', '/content/drive/MyDrive/TorNet_Backup/packages/tornet_detection-0.1.6-py3-none-any.whl'], returncode=0)

In [4]:
import tornado_detection

if (
    tornado_detection.__version__
    != PACKAGE_VERSION
):
    raise RuntimeError(
        "Unexpected installed package version: "
        f"{tornado_detection.__version__} != "
        f"{PACKAGE_VERSION}"
    )

print(
    "tornado_detection version:",
    tornado_detection.__version__,
)


tornado_detection version: 0.1.6


In [5]:
from tornado_detection.data import (
    assign_model_splits,
    load_canonical_frame_index,
)

frame_index = load_canonical_frame_index(
    MANIFESTS_ROOT
)
assigned_frame_index = assign_model_splits(
    frame_index,
    validation_fraction=0.20,
    seed=20260913,
)

year_index = (
    assigned_frame_index.loc[
        assigned_frame_index[
            "year"
        ].eq(2013)
    ]
    .sort_values(
        [
            "archive_member",
            "frame_index",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

assert len(year_index) == 16_284
assert (
    year_index["file_id"].nunique()
    == 4_071
)

print(
    "2013 frames:",
    f"{len(year_index):,}",
)
print(
    "2013 files:",
    f"{year_index['file_id'].nunique():,}",
)
print(
    "2013 positive frames:",
    f"{int(year_index['frame_label'].sum()):,}",
)
print()
print(
    year_index.groupby(
        "model_split",
        sort=False,
    )
    .agg(
        frames=("frame_id", "size"),
        files=("file_id", "nunique"),
        positives=("frame_label", "sum"),
    )
    .to_string()
)


2013 frames: 16,284
2013 files: 4,071
2013 positive frames: 745

             frames  files  positives
model_split                          
test           2292    573        157
train         11056   2764        445
validation     2936    734        143


In [6]:
import shutil
import time

if LOCAL_ARCHIVE_PATH.exists():
    LOCAL_ARCHIVE_PATH.unlink()

if EXTRACTION_ROOT.exists():
    shutil.rmtree(EXTRACTION_ROOT)

usage = shutil.disk_usage("/content")
archive_size = (
    DRIVE_ARCHIVE_PATH.stat().st_size
)

if usage.free < archive_size * 3:
    raise RuntimeError(
        "Insufficient local space for archive "
        "copy and extraction"
    )

copy_started = time.perf_counter()

shutil.copyfile(
    DRIVE_ARCHIVE_PATH,
    LOCAL_ARCHIVE_PATH,
)

copy_seconds = (
    time.perf_counter()
    - copy_started
)

local_size = (
    LOCAL_ARCHIVE_PATH.stat().st_size
)

if local_size != archive_size:
    raise RuntimeError(
        "Local archive size differs from Drive: "
        f"{local_size} != {archive_size}"
    )

copy_mib_per_second = (
    local_size
    / (1024 ** 2)
    / copy_seconds
)

print(
    "archive bytes:",
    local_size,
)
print(
    "copy seconds:",
    round(copy_seconds, 3),
)
print(
    "copy MiB/s:",
    round(copy_mib_per_second, 3),
)


archive bytes: 3159866899
copy seconds: 41.345
copy MiB/s: 72.886


In [7]:
import tarfile

expected_members = set(
    year_index[
        "archive_member"
    ].unique()
)

EXTRACTION_ROOT.mkdir(
    parents=True,
    exist_ok=False,
)
extraction_root_resolved = (
    EXTRACTION_ROOT.resolve()
)

extracted_members = set()
extracted_bytes = 0
extraction_started = (
    time.perf_counter()
)

with tarfile.open(
    LOCAL_ARCHIVE_PATH,
    mode="r:gz",
) as archive:
    for member in archive:
        if (
            not member.isfile()
            or member.name
            not in expected_members
        ):
            continue

        destination = (
            EXTRACTION_ROOT
            / member.name
        )
        destination_resolved = (
            destination.resolve()
        )

        if (
            extraction_root_resolved
            not in destination_resolved.parents
        ):
            raise RuntimeError(
                "Unsafe archive member path: "
                f"{member.name}"
            )

        destination.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        source_file = archive.extractfile(
            member
        )

        if source_file is None:
            raise RuntimeError(
                "Could not read archive member: "
                f"{member.name}"
            )

        with (
            source_file,
            destination.open("wb")
            as output_file,
        ):
            shutil.copyfileobj(
                source_file,
                output_file,
                length=1024 * 1024,
            )

        actual_size = (
            destination.stat().st_size
        )

        if actual_size != member.size:
            raise RuntimeError(
                "Extracted size mismatch for "
                f"{member.name}: "
                f"{actual_size} != {member.size}"
            )

        extracted_members.add(
            member.name
        )
        extracted_bytes += actual_size

extraction_seconds = (
    time.perf_counter()
    - extraction_started
)

missing_members = sorted(
    expected_members
    - extracted_members
)
unexpected_members = sorted(
    extracted_members
    - expected_members
)

if missing_members:
    raise RuntimeError(
        "Missing canonical archive members: "
        f"{missing_members[:10]}"
    )

if unexpected_members:
    raise RuntimeError(
        "Extracted unexpected archive members: "
        f"{unexpected_members[:10]}"
    )

if len(extracted_members) != 4_071:
    raise RuntimeError(
        "Unexpected extracted member count: "
        f"{len(extracted_members)}"
    )

print(
    "extracted files:",
    f"{len(extracted_members):,}",
)
print(
    "extracted GiB:",
    round(
        extracted_bytes
        / (1024 ** 3),
        3,
    ),
)
print(
    "extraction seconds:",
    round(extraction_seconds, 3),
)
print(
    "extracted files/s:",
    round(
        len(extracted_members)
        / extraction_seconds,
        3,
    ),
)


extracted files: 4,071
extracted GiB: 3.141
extraction seconds: 52.249
extracted files/s: 77.915


In [8]:
import numpy as np
import xarray as xr

from tornado_detection.data import (
    build_frame_tensor,
)

frames_processed = 0
positive_frames = 0
finite_values = 0
total_values = 0
tensor_bytes = 0
label_mismatches = []

loading_started = time.perf_counter()

for (
    archive_member,
    member_frames,
) in year_index.groupby(
    "archive_member",
    sort=False,
):
    netcdf_path = (
        EXTRACTION_ROOT
        / archive_member
    )

    if not netcdf_path.is_file():
        raise FileNotFoundError(
            f"Extracted file missing: "
            f"{netcdf_path}"
        )

    with xr.open_dataset(
        netcdf_path,
        engine="netcdf4",
    ) as dataset:
        for row in (
            member_frames.itertuples(
                index=False
            )
        ):
            result = build_frame_tensor(
                dataset,
                int(row.frame_index),
            )

            expected_label = int(
                row.frame_label
            )

            if (
                result.label
                != expected_label
            ):
                label_mismatches.append(
                    {
                        "frame_id": row.frame_id,
                        "manifest_label": (
                            expected_label
                        ),
                        "netcdf_label": (
                            result.label
                        ),
                    }
                )

            frames_processed += 1
            positive_frames += (
                result.label
            )
            finite_values += int(
                np.isfinite(
                    result.values
                ).sum()
            )
            total_values += int(
                result.values.size
            )
            tensor_bytes += int(
                result.values.nbytes
            )

loading_seconds = (
    time.perf_counter()
    - loading_started
)

if label_mismatches:
    raise AssertionError(
        "Manifest/NetCDF label mismatches: "
        f"{label_mismatches[:10]}"
    )

if frames_processed != 16_284:
    raise AssertionError(
        "Unexpected processed frame count: "
        f"{frames_processed}"
    )

expected_positive_frames = int(
    year_index[
        "frame_label"
    ].sum()
)

if (
    positive_frames
    != expected_positive_frames
):
    raise AssertionError(
        "Unexpected positive-frame count: "
        f"{positive_frames} != "
        f"{expected_positive_frames}"
    )

frames_per_second = (
    frames_processed
    / loading_seconds
)
files_per_second = (
    len(extracted_members)
    / loading_seconds
)
effective_mib_per_second = (
    tensor_bytes
    / (1024 ** 2)
    / loading_seconds
)
finite_fraction = (
    finite_values
    / total_values
)

print(
    "frames processed:",
    f"{frames_processed:,}",
)
print(
    "positive frames:",
    f"{positive_frames:,}",
)
print(
    "label mismatches:",
    len(label_mismatches),
)
print(
    "loading seconds:",
    round(loading_seconds, 3),
)
print(
    "frames/s:",
    round(frames_per_second, 3),
)
print(
    "files/s:",
    round(files_per_second, 3),
)
print(
    "effective tensor MiB/s:",
    round(
        effective_mib_per_second,
        3,
    ),
)
print(
    "overall finite fraction:",
    round(finite_fraction, 6),
)


frames processed: 16,284
positive frames: 745
label mismatches: 0
loading seconds: 197.848
frames/s: 82.306
files/s: 20.576
effective tensor MiB/s: 36.17
overall finite fraction: 0.59181


In [9]:
import pandas as pd

benchmark_summary = pd.DataFrame(
    [
        {
            "year": 2013,
            "archive_gib": (
                local_size / (1024 ** 3)
            ),
            "copy_seconds": (
                copy_seconds
            ),
            "copy_mib_per_second": (
                copy_mib_per_second
            ),
            "extracted_files": (
                len(extracted_members)
            ),
            "extracted_gib": (
                extracted_bytes
                / (1024 ** 3)
            ),
            "extraction_seconds": (
                extraction_seconds
            ),
            "frames_processed": (
                frames_processed
            ),
            "loading_seconds": (
                loading_seconds
            ),
            "frames_per_second": (
                frames_per_second
            ),
            "files_per_second": (
                files_per_second
            ),
            "effective_tensor_mib_per_second": (
                effective_mib_per_second
            ),
            "finite_fraction": (
                finite_fraction
            ),
            "label_mismatches": (
                len(label_mismatches)
            ),
        }
    ]
)

print(
    benchmark_summary.to_string(
        index=False
    )
)


 year  archive_gib  copy_seconds  copy_mib_per_second  extracted_files  extracted_gib  extraction_seconds  frames_processed  loading_seconds  frames_per_second  files_per_second  effective_tensor_mib_per_second  finite_fraction  label_mismatches
 2013     2.942855     41.344928            72.886423             4071       3.141296            52.24918             16284       197.847539            82.3058          20.57645                        36.169541          0.59181                 0


In [10]:
shutil.rmtree(
    EXTRACTION_ROOT
)
LOCAL_ARCHIVE_PATH.unlink()

assert not EXTRACTION_ROOT.exists()
assert not LOCAL_ARCHIVE_PATH.exists()

print(
    "Removed all Colab-local benchmark artifacts"
)


Removed all Colab-local benchmark artifacts
